# LINet OneCycleLR HPO on SUN RGB-D - Google Colab

**Two-part workflow:**
1. **LR Finder** — Leslie Smith sweep to find `base_max_lr`
2. **HPO with OneCycleLR** — Ray Tune + Optuna (no ASHA) using the anchored `base_max_lr`

---

## Checklist Before Running:

- [ ] **Enable A100 GPU:** Runtime > Change runtime type > A100
- [ ] **Upload dataset to Drive:** `MyDrive/datasets/sunrgbd_19_traintest.tar.gz`

## 1. Environment Setup & GPU Verification

In [1]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

# Check PyTorch and CUDA
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    # Check if it's A100
    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\n✅ A100 GPU detected - PERFECT for training!")
    elif 'V100' in gpu_name:
        print("\n✅ V100 GPU detected - Good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\n⚠️  T4 GPU detected - Will be slower, consider upgrading to A100")
    else:
        print(f"\n⚠️  GPU: {gpu_name} - Consider using A100 for best performance")
else:
    print("\n❌ NO GPU DETECTED!")
    print("Please enable GPU: Runtime → Change runtime type → Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

GPU VERIFICATION
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU Device: NVIDIA A100-SXM4-80GB
GPU Memory: 79.25 GB

✅ A100 GPU detected - PERFECT for training!



In [2]:
# Detailed GPU info
!nvidia-smi

Wed Mar 25 16:50:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             53W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [3]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

Mounted at /content/drive

✅ Google Drive mounted successfully!

Drive contents:
total 3117883
-rw------- 1 root root        176 Sep 21  2019 06-lab2.gdoc
-rw------- 1 root root      21621 Sep 30  2024 113-1363667-3121001@USSR24093000064918@pre-paid.png
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (1).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (2).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (3).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final.gdoc
-rw------- 1 root root        176 Jul 11  2025 2025_Gabriel_Clinger_Contractor Agreement_BASE copy.gdoc
-rw------- 1 root root      32204 Apr 18  2022 2900 On First- Welcome Home Next Steps.docx
-rw------- 1 root root       8822 Jun 24  2017 A6.docx
-rw------- 1 root root      22204 Jan 21  2023 activity (1).xlsx
-rw------- 1 root root      22161 Jan 21  2023 activity (2).xlsx
-rw------- 1 root root        176 Jan 21  2023 activity.gsheet
-rw------- 

## 3. Clone Repository to Local Disk (Fast I/O)

**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

**Default:** Clone from GitHub (recommended - always gets latest code)

In [4]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"  # UPDATE THIS
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"  # Local copy for fast I/O

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

# Ensure we're in a valid directory
os.chdir('/content')
print(f"Starting in: {os.getcwd()}")

# Check if repo already exists (same session, rerunning cell)
if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"\n📁 Repo already exists: {LOCAL_REPO_PATH}")
    print(f"🔄 Pulling latest changes...")

    os.chdir(LOCAL_REPO_PATH)
    !git pull
    print("✅ Repo updated")

# Clone from GitHub (first run)
else:
    # Remove old incomplete copy if exists
    if Path(LOCAL_REPO_PATH).exists():
        print(f"\n🗑️  Removing incomplete repo copy...")
        !rm -rf {LOCAL_REPO_PATH}

    print(f"\n🔄 Cloning from GitHub...")
    print(f"   Repo: {GITHUB_REPO}")
    print(f"   Destination: {LOCAL_REPO_PATH}")

    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}

    # Verify clone succeeded
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository to {LOCAL_REPO_PATH}")

    print("✅ Repo cloned successfully")
    os.chdir(LOCAL_REPO_PATH)

# Verify repo structure
print(f"\n📂 Repository structure:")
!ls -la {LOCAL_REPO_PATH}

print(f"\n✅ Working directory: {os.getcwd()}")

REPOSITORY SETUP
Starting in: /content

🔄 Cloning from GitHub...
   Repo: https://github.com/clingergab/Multi-Stream-Neural-Networks.git
   Destination: /content/Multi-Stream-Neural-Networks
Cloning into '/content/Multi-Stream-Neural-Networks'...
remote: Enumerating objects: 3159, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 3159 (delta 82), reused 75 (delta 75), pack-reused 3072 (from 2)
Receiving objects: 100% (3159/3159), 113.47 MiB | 29.75 MiB/s, done.
Resolving deltas: 100% (1983/1983), done.
Encountered 49 file(s) that should have been pointers, but weren't:
	tests/augmentation_comparison.png
	tests/augmentation_test.png
	tests/balanced_augmentation_test.png
	tests/balanced_samples_comparison.png
	tests/dataset_orthogonal_loading.png
	tests/decaying_restarts_eta_min_bug.png
	tests/easing_formula_analysis.png
	tests/easing_schedulers_comparison.png
	tests/global_vs_local_comparison.png
	tests/linear_scale_compar

## 4. Install Dependencies

In [5]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] optuna kornia

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import ray
import kornia

print("✅ All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   ray: {ray.__version__}")
print(f"   kornia: {kornia.__version__}")


Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 MB 34.9 MB/s eta 0:00:00
✅ All dependencies installed!
   h5py: 3.16.0
   matplotlib: 3.10.0
   ray: 2.54.0
   kornia: 0.8.2


## 5. Copy SUN RGB-D Dataset to Local Disk

**Performance Note:** Local disk I/O is ~10-20x faster than Drive!

**Dataset:** SUN RGB-D 19-category preprocessed dataset with RGB + Depth


In [6]:
from pathlib import Path
import os

# Paths
DRIVE_DATASET_TAR = "/content/drive/MyDrive/datasets/sunrgbd_19_traintest.tar.gz"
LOCAL_DATASET_PATH = "/dev/shm/sunrgbd_19_traintest"

print("=" * 60)
print("SUN RGB-D 19-CATEGORY DATASET SETUP")
print("=" * 60)

if Path(LOCAL_DATASET_PATH).exists():
    print(f"Already on local disk: {LOCAL_DATASET_PATH}")
    train_count = len(list(Path(f"{LOCAL_DATASET_PATH}/train/rgb").glob("*.png")))
    print(f"   Train samples: {train_count}")
elif Path(DRIVE_DATASET_TAR).exists():
    print(f"Found on Drive: {DRIVE_DATASET_TAR}")
    tar_name = Path(DRIVE_DATASET_TAR).name
    local_tar = f"/dev/shm/{tar_name}"
    !rsync -ah --info=progress2 {DRIVE_DATASET_TAR} {local_tar}
    print(f"\nExtracting...")
    !tar -xzf {local_tar} -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"
    !rm {local_tar}
    train_count = len(list(Path(f"{LOCAL_DATASET_PATH}/train/rgb").glob("*.png")))
    print(f"Extracted. Train samples: {train_count}")
else:
    raise FileNotFoundError(f"Dataset not found at {DRIVE_DATASET_TAR}")

print(f"\nDataset ready at: {LOCAL_DATASET_PATH}")


SUN RGB-D 19-CATEGORY DATASET SETUP
Found on Drive: /content/drive/MyDrive/datasets/sunrgbd_19_traintest.tar.gz
          1.54G 100%   52.84MB/s    0:00:27 (xfr#1, to-chk=0/1)

Extracting...
Extracted. Train samples: 0

Dataset ready at: /dev/shm/sunrgbd_19_traintest


## 6. Setup Python Path & Import LINet


In [ ]:
import sys
import os

modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project structure:")
!ls -la {project_root}/src/models/

print("\nImporting LiNet and dataloaders...")
from src.models.linear_integration.li_net3 import li_resnet18
from src.data_utils.sunrgbd_dataset import get_sunrgbd_dataloaders, SUNRGBDDataset
from src.training.augmentation_config import AugmentationConfig

from ray import train, tune

print("All imports successful!")

---

# Part 1: Learning Rate Finder (Leslie Smith Method)

**Goal:** Find the optimal `base_max_lr` for OneCycleLR by sweeping the learning rate
exponentially from ~1e-7 to ~10.0 over a few hundred batches.

**How to read the plot:**
- Look for the **steepest downward slope** — NOT the lowest point
- The steepest descent is where the network learns fastest without destabilizing
- Pick the LR in the middle of that slope as your `base_max_lr`
- Heuristic: find where loss diverges and divide that LR by 10

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
from collections import Counter
from sklearn.model_selection import train_test_split
from torch.amp import autocast, GradScaler

from src.models.linear_integration.li_net3 import li_resnet18
from src.training.optimizers import create_stream_optimizer
from src.data_utils.sunrgbd_dataset import SUNRGBDDataset, _load_norm_stats
from src.training.augmentation_config import AugmentationConfig
from src.utils.seed import set_seed

# =============================================
# LR FINDER CONFIGURATION
# =============================================
SEED = 42
LR_START = 1e-7
LR_END = 10.0
NUM_BATCHES = 300
SMOOTH_FACTOR = 0.1       # EMA smoothing (higher = more responsive)
DIVERGE_THRESHOLD = 4.0   # Stop when loss > threshold * min_loss

set_seed(SEED, deterministic=False)
g = torch.Generator().manual_seed(SEED)

# Default augmentation (no tuning for LR finder)
aug_config = AugmentationConfig()

# Load data — same setup as HPO trials
norm_stats = _load_norm_stats(LOCAL_DATASET_PATH)

train_dataset = SUNRGBDDataset(
    data_root=LOCAL_DATASET_PATH,
    split='train',
    normalize=False,
    **aug_config.to_dict(),
)

# 80/20 stratified split (same as HPO)
all_labels = train_dataset.labels
train_indices, _ = train_test_split(
    list(range(len(all_labels))),
    test_size=0.2,
    random_state=SEED,
    stratify=all_labels,
)
train_subset = torch.utils.data.Subset(train_dataset, train_indices)

# Weighted sampler (class-balanced, same as HPO)
subset_labels = [all_labels[i] for i in train_indices]
label_counts = Counter(subset_labels)
num_samples = len(subset_labels)
class_weights = {l: num_samples / c for l, c in label_counts.items()}
sample_weights = torch.tensor([class_weights[l] for l in subset_labels], dtype=torch.float32)
train_sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights,
    num_samples=num_samples,
    replacement=True,
    generator=g,
)

train_loader = torch.utils.data.DataLoader(
    train_subset,
    batch_size=64,
    shuffle=False,
    sampler=train_sampler,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=2,
)

# Create fresh model
model = li_resnet18(
    num_classes=19,
    stream_input_channels=[3, 1],
    dropout_p=0.3,
    width_multiplier=0.75,
    device="cuda",
    use_amp=True,
)

# Uniform LR for all groups (stem_lr_multiplier=1.0 — no DLR for finder)
optimizer = create_stream_optimizer(
    model,
    optimizer_type='adamw',
    stream_lrs=LR_START,
    stream_weight_decays=1e-4,
    shared_lr=LR_START,
    integration_weight_decay=1e-4,
    stem_lr_multiplier=1.0,
)

# Compile model for GPU augmentation (no scheduler — we control LR manually)
model.compile(
    optimizer=optimizer,
    scheduler=None,
    loss='cross_entropy',
    label_smoothing=0.1,
    gpu_augmentation=True,
    norm_stats=norm_stats,
    **aug_config.to_dict(),
)

# LR multiplier per step: LR_START * mult^NUM_BATCHES = LR_END
mult = (LR_END / LR_START) ** (1.0 / NUM_BATCHES)

# Run LR sweep
lrs = []
losses = []
smoothed_loss = 0.0
min_loss = float('inf')
batch_iter = iter(train_loader)

model.train()
scaler = GradScaler('cuda')

for batch_idx in range(NUM_BATCHES):
    # Get batch (cycle through loader if needed)
    try:
        batch_data = next(batch_iter)
    except StopIteration:
        batch_iter = iter(train_loader)
        batch_data = next(batch_iter)

    # Unpack: DataLoader returns (rgb, depth, labels)
    *stream_batches, targets = batch_data
    stream_batches = [b.cuda(non_blocking=True) for b in stream_batches]
    targets = targets.cuda(non_blocking=True)

    # GPU augmentation
    if model.gpu_aug is not None:
        stream_batches[0], stream_batches[1] = model.gpu_aug(
            stream_batches[0], stream_batches[1]
        )

    # Forward + backward with AMP
    optimizer.zero_grad()
    with autocast('cuda'):
        outputs = model(stream_batches)
        loss = model.criterion(outputs, targets)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    # Record
    current_lr = optimizer.param_groups[0]['lr']
    current_loss = loss.item()

    # Exponential moving average of loss
    if batch_idx == 0:
        smoothed_loss = current_loss
    else:
        smoothed_loss = SMOOTH_FACTOR * current_loss + (1 - SMOOTH_FACTOR) * smoothed_loss

    lrs.append(current_lr)
    losses.append(smoothed_loss)

    # Track minimum
    if smoothed_loss < min_loss:
        min_loss = smoothed_loss

    # Stop if diverging
    if batch_idx > 10 and smoothed_loss > DIVERGE_THRESHOLD * min_loss:
        print(f"Stopping at batch {batch_idx}: loss diverged "
              f"({smoothed_loss:.4f} > {DIVERGE_THRESHOLD} * {min_loss:.4f})")
        break

    # Increase LR for next step
    for pg in optimizer.param_groups:
        pg['lr'] *= mult

print(f"Swept {len(lrs)} batches, LR range: {lrs[0]:.2e} -> {lrs[-1]:.2e}")

# Clean up model to free GPU memory
del model, optimizer, scaler
torch.cuda.empty_cache()

In [ ]:
# =============================================
# PLOT LR FINDER RESULTS
# =============================================

fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax.plot(lrs, losses, linewidth=1.5)
ax.set_xscale('log')
ax.set_xlabel('Learning Rate (log scale)', fontsize=12)
ax.set_ylabel('Smoothed Loss', fontsize=12)
ax.set_title('LR Finder: Loss vs Learning Rate', fontsize=14)
ax.grid(True, alpha=0.3)

# Find steepest descent (max negative gradient in log-lr space)
log_lrs = np.log10(lrs)
loss_arr = np.array(losses)
gradients = np.gradient(loss_arr, log_lrs)
steepest_idx = np.argmin(gradients)
suggested_lr = lrs[steepest_idx]

ax.axvline(x=suggested_lr, color='r', linestyle='--', alpha=0.7,
           label=f'Steepest descent: {suggested_lr:.2e}')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"\nSuggested base_max_lr (steepest descent): {suggested_lr:.2e}")
print("\nVisually inspect the plot:")
print("  1. Find the steepest downward slope")
print("  2. Pick the LR in the MIDDLE of that slope")
print("  3. Enter it as BASE_MAX_LR in the next cell")

In [ ]:
# =============================================
# SET BASE_MAX_LR FROM THE LR FINDER PLOT
# =============================================
# EDIT THIS VALUE based on the plot above.
# Pick the LR at the steepest downward slope.

BASE_MAX_LR = 3e-3  # <-- EDIT THIS

print(f"Using base_max_lr = {BASE_MAX_LR:.2e}")
print(f"  Initial LR (div_factor=25): {BASE_MAX_LR / 25:.2e}")
print(f"  Final LR (div=25, final_div=1e4): {BASE_MAX_LR / 25 / 1e4:.2e}")

---

# Part 2: Hyperparameter Optimization with OneCycleLR

**Strategy:**
- `BASE_MAX_LR` is **fixed** from Part 1 (the LR Finder)
- HPO tunes `stem_lr_multiplier`, regularization, augmentation, and OneCycle params
- OneCycleLR automatically scales each parameter group's entire cycle
- **No ASHA pruning** — OneCycle's mid-run chaos would cause ASHA to kill optimal trials

**OneCycle params derived from `BASE_MAX_LR`:**
- `initial_lr = BASE_MAX_LR / div_factor` (warmup start)
- `max_lr = BASE_MAX_LR` (peak, scaled by `stem_lr_multiplier` for stem groups)
- `final_lr = initial_lr / final_div_factor` (cooldown end)

In [8]:
import os
import time

# 1. Define Paths explicitly
mps_pipe_dir = "/tmp/nvidia-mps"
mps_log_dir = "/tmp/nvidia-log"

# 2. Create the directories (CRITICAL: Daemon fails if log dir doesn't exist)
os.makedirs(mps_pipe_dir, exist_ok=True)
os.makedirs(mps_log_dir, exist_ok=True)

# 3. Set Environment Variables for the current Python process
os.environ["CUDA_MPS_PIPE_DIRECTORY"] = mps_pipe_dir
os.environ["CUDA_MPS_LOG_DIRECTORY"] = mps_log_dir
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# 4. Kill any stale MPS daemon before starting fresh
print("Stopping any existing MPS daemon...")
!echo quit | nvidia-cuda-mps-control 2>/dev/null; sleep 1

# 5. Configure GPU and Start Daemon
print("Setting GPU to Exclusive Process Mode...")
!nvidia-smi -i 0 -c EXCLUSIVE_PROCESS

print("Starting MPS Daemon...")
!export CUDA_MPS_PIPE_DIRECTORY={mps_pipe_dir} && \
 export CUDA_MPS_LOG_DIRECTORY={mps_log_dir} && \
 nvidia-cuda-mps-control -d

# 6. Verify it is running
print("Verifying Daemon Status...")
time.sleep(2)
!ps -ef | grep [m]ps

# Check if the pipe file actually exists
if os.path.exists(os.path.join(mps_pipe_dir, "control")):
    print("\u2705 MPS Control Pipe found. Setup success.")
else:
    print("\u274c MPS Control Pipe NOT found. Check /tmp/nvidia-log for errors.")
    !cat {mps_log_dir}/control.log

Setting GPU to Exclusive Process Mode...
Set compute mode to EXCLUSIVE_PROCESS for GPU 00000000:00:05.0.
All done.
Starting MPS Daemon...
Verifying Daemon Status...
root        1997       1  0 16:54 ?        00:00:00 nvidia-cuda-mps-control -d
root        2003     588  0 16:54 ?        00:00:00 /bin/bash -c ps -ef | grep mps
root        2005    2003  0 16:54 ?        00:00:00 grep mps
✅ MPS Control Pipe found. Setup success.


In [ ]:
import random
import numpy as np

import ray
from ray import tune
from ray.tune.search.optuna import OptunaSearch
from optuna.samplers import TPESampler
import torch
from collections import Counter
from sklearn.model_selection import train_test_split

from src.models.linear_integration.li_net3 import li_resnet18
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler
from src.data_utils.sunrgbd_dataset import SUNRGBDDataset, _load_norm_stats
from src.training.augmentation_config import AugmentationConfig
from src.utils.seed import set_seed


class RayTuneReporter:
    """Callback for reporting metrics to Ray Tune during training."""

    def __init__(self):
        self.best_accuracy = 0.0
        self.best_loss = float('inf')
        self.best_train_acc = 0.0
        self.best_val_mca = 0.0
        self.best_train_mca = 0.0
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs):
        """Report current AND best metrics to Ray Tune."""
        if logs['val_accuracy'] > self.best_accuracy:
            self.best_accuracy = logs['val_accuracy']
            if logs['train_accuracy'] > self.best_train_acc:
                self.best_train_acc = logs['train_accuracy']

        if logs['val_loss'] < self.best_loss:
            self.best_loss = logs['val_loss']

        val_mca = logs.get('val_mca', 0.0)
        train_mca = logs.get('train_mca', 0.0)
        if val_mca > self.best_val_mca:
            self.best_val_mca = val_mca
            self.best_epoch = epoch
            if train_mca > self.best_train_mca:
                self.best_train_mca = train_mca

        gap = self.best_train_mca - self.best_val_mca
        composite = self.best_val_mca - 5 * (gap**3)

        tune.report({
            "accuracy": logs['val_accuracy'],
            "loss": logs['val_loss'],
            "best_accuracy": self.best_accuracy,
            "best_loss": self.best_loss,
            "train_loss": logs['train_loss'],
            "train_accuracy": logs['train_accuracy'],
            "best_train_acc": self.best_train_acc,
            "val_mca": val_mca,
            "train_mca": train_mca,
            "best_val_mca": self.best_val_mca,
            "best_train_mca": self.best_train_mca,
            "composite": composite,
            "best_epoch": self.best_epoch,
        })


def train_linet_onecycle(
    config,
    data_root=None,
    norm_stats=None,
    base_max_lr=None,
    seed=42,
):
    """
    Trainable function for Ray Tune — OneCycleLR with locked 80/20 stratified split.

    Args:
        config: Ray Tune configuration dict with hyperparameters
        data_root: Path to dataset root (with train/ directory)
        norm_stats: Normalization statistics dict
        base_max_lr: Maximum learning rate from LR Finder (fixed anchor)
        seed: Random seed for reproducible trials
    """
    set_seed(seed, deterministic=False)
    g = torch.Generator().manual_seed(seed)

    EPOCHS = 50

    # Per-trial augmentation config
    aug_config = AugmentationConfig(
        rgb_aug_prob=config.get("rgb_aug_prob"),
        rgb_aug_mag=config.get("rgb_aug_mag"),
        depth_aug_prob=config.get("depth_aug_prob"),
        depth_aug_mag=config.get("depth_aug_mag"),
    )

    # Two dataset instances from the same train/ directory
    train_dataset = SUNRGBDDataset(
        data_root=data_root,
        split='train',
        normalize=False,
        **aug_config.to_dict(),
    )
    val_dataset = SUNRGBDDataset(
        data_root=data_root,
        split='train',
        normalize=False,
    )
    val_dataset.split = 'val'

    # Locked 80/20 stratified split (deterministic)
    all_labels = train_dataset.labels
    train_indices, val_indices = train_test_split(
        list(range(len(all_labels))),
        test_size=0.2,
        random_state=seed,
        stratify=all_labels,
    )

    train_subset = torch.utils.data.Subset(train_dataset, train_indices)
    val_subset = torch.utils.data.Subset(val_dataset, val_indices)

    # Stratified sampling for training
    subset_labels = [all_labels[i] for i in train_indices]
    label_counts = Counter(subset_labels)
    num_samples = len(subset_labels)
    class_weights = {label: num_samples / count for label, count in label_counts.items()}
    sample_weights = torch.tensor(
        [class_weights[label] for label in subset_labels], dtype=torch.float32
    )

    train_sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights,
        num_samples=num_samples,
        replacement=True,
        generator=g,
    )

    def worker_init_fn(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    train_loader = torch.utils.data.DataLoader(
        train_subset,
        batch_size=64,
        shuffle=False,
        sampler=train_sampler,
        num_workers=2,
        prefetch_factor=4,
        persistent_workers=True,
        pin_memory=True,
        worker_init_fn=worker_init_fn,
    )
    val_loader = torch.utils.data.DataLoader(
        val_subset,
        batch_size=64,
        shuffle=False,
        num_workers=1,
        prefetch_factor=2,
        persistent_workers=False,
        pin_memory=True,
        worker_init_fn=worker_init_fn,
    )

    # Create Model
    model = li_resnet18(
        num_classes=19,
        stream_input_channels=[3, 1],
        dropout_p=config["dropout_p"],
        width_multiplier=0.75,
        device="cuda",
        use_amp=True,
    )

    # OneCycle-specific parameter setup
    stem_mult = config["stem_lr_multiplier"]
    div_factor = config["div_factor"]

    # Initial LR = max_lr / div_factor (OneCycleLR sets this automatically,
    # but we set it here so optimizer param groups start consistent)
    initial_lr = base_max_lr / div_factor

    # Create Optimizer with DLR parameter groups
    optimizer = create_stream_optimizer(
        model,
        optimizer_type='adamw',
        stream_lrs=initial_lr,
        stream_weight_decays=config["wd"],
        shared_lr=initial_lr,
        integration_weight_decay=config["wd"],
        stem_lr_multiplier=stem_mult,
    )

    # Build per-group max_lr list for OneCycleLR
    # When stem_mult != 1.0: 6 groups [stem.0, stem.1, backbone.0, backbone.1, integration, other]
    # When stem_mult == 1.0: 4 groups [backbone.0, backbone.1, integration, other]
    n_groups = len(optimizer.param_groups)
    if n_groups == 6:
        max_lr_list = [
            base_max_lr * stem_mult,  # stem.0 (RGB conv1)
            base_max_lr * stem_mult,  # stem.1 (Depth conv1)
            base_max_lr,              # backbone.0 (RGB layers)
            base_max_lr,              # backbone.1 (Depth layers)
            base_max_lr,              # integration + classifier
            base_max_lr,              # other (BN params)
        ]
    else:
        max_lr_list = [base_max_lr] * n_groups

    # Create OneCycleLR scheduler
    # No separate warmup — OneCycleLR's pct_start handles warmup internally
    scheduler = setup_scheduler(
        optimizer,
        scheduler_type='onecycle',
        epochs=EPOCHS,
        train_loader_len=len(train_loader),
        max_lr=max_lr_list,
        pct_start=config["pct_start"],
        div_factor=config["div_factor"],
        final_div_factor=config.get("final_div_factor", 1e4),
    )

    # Compile
    model.compile(
        optimizer=optimizer,
        scheduler=scheduler,
        loss='cross_entropy',
        label_smoothing=config["label_smoothing"],
        gpu_augmentation=True,
        norm_stats=norm_stats,
        **aug_config.to_dict(),
    )

    # Train — model.fit() auto-detects OneCycleLR and steps per batch
    model.fit(
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=EPOCHS,
        early_stopping=False,
        monitor='val_mca',
        patience=10,
        grad_clip_norm=config["grad_clip_norm"],
        callbacks=[RayTuneReporter()],
        verbose=False,
    )

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
# Ray Tune saves ALL experiment state to DRIVE_STORAGE_PATH via storage_path.
# When Colab dies, re-run the notebook — Tuner.restore() picks up where it
# left off. Completed trials preserved, interrupted trials restart.
# =============================================================================

import os
import pandas as pd
from pathlib import Path

# --- Ray Tune persistent storage on Google Drive ---
DRIVE_STORAGE_PATH = "/content/drive/MyDrive/ray_tune_experiments"
LOCAL_STORAGE_PATH = "/content/ray_results"
EXPERIMENT_NAME = "sun_rgbd_onecycle_hpo"

SEED = 42
NUM_SAMPLES = 300  # No ASHA pruning — every trial runs to completion
EPOCHS = 50

Path(DRIVE_STORAGE_PATH).mkdir(parents=True, exist_ok=True)
Path(LOCAL_STORAGE_PATH).mkdir(parents=True, exist_ok=True)

experiment_path = os.path.join(DRIVE_STORAGE_PATH, EXPERIMENT_NAME)
local_experiment_path = os.path.join(LOCAL_STORAGE_PATH, EXPERIMENT_NAME)
RESUME_EXISTING = os.path.exists(experiment_path)

if RESUME_EXISTING:
    _exp_files = os.listdir(experiment_path) if os.path.isdir(experiment_path) else []
    if len(_exp_files) == 0:
        print(f"  WARNING: {experiment_path} exists but is empty -- starting fresh")
        RESUME_EXISTING = False

print(f"Drive storage: {DRIVE_STORAGE_PATH}")
print(f"Local storage: {LOCAL_STORAGE_PATH}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Resume existing: {RESUME_EXISTING}")
print(f"Total trials: {NUM_SAMPLES}")
print(f"Epochs per trial: {EPOCHS}")
print(f"base_max_lr: {BASE_MAX_LR:.2e}")
print(f"Scheduler: OneCycleLR (no ASHA pruning)")

if RESUME_EXISTING:
    print(f"\n  Previous experiment found at {experiment_path}")
    print(f"  Copying Drive -> local, then Tuner.restore() from local.")
    import subprocess as _sp_cfg
    os.makedirs(local_experiment_path, exist_ok=True)
    _result = _sp_cfg.run(
        ["rsync", "-a", experiment_path + "/", local_experiment_path + "/"],
        capture_output=True, text=True,
    )
    if _result.returncode == 0:
        print(f"  Restored to {local_experiment_path}")
    else:
        raise RuntimeError(f"Restore failed: {_result.stderr[:300]}")

In [ ]:
# Initialize Ray
import shutil
import subprocess
import time as _time

from ray.tune import CLIReporter
from ray.tune import Callback as TuneCallback

os.environ["RAY_AIR_NEW_OUTPUT"] = "0"


class DriveSyncCallback(TuneCallback):
    """Periodically rsyncs local Ray Tune experiment state to Google Drive."""

    def __init__(self, local_storage_path, drive_storage_path, experiment_name,
                 sync_interval_seconds=300):
        self._local_path = os.path.join(local_storage_path, experiment_name)
        self._drive_path = os.path.join(drive_storage_path, experiment_name)
        self._sync_interval = sync_interval_seconds
        self._last_sync = 0.0

    def _sync(self, reason=""):
        if not os.path.isdir(self._local_path):
            return
        try:
            os.makedirs(self._drive_path, exist_ok=True)
            result = subprocess.run(
                ["rsync", "-a",
                 self._local_path + "/",
                 self._drive_path + "/"],
                capture_output=True, text=True, timeout=120,
            )
            if result.returncode == 0:
                self._last_sync = _time.time()
                print(f"[DriveSyncCallback] synced to Drive ({reason})")
            else:
                print(f"[DriveSyncCallback] WARNING: rsync failed: {result.stderr[:200]}")
        except subprocess.TimeoutExpired:
            print(f"[DriveSyncCallback] WARNING: rsync timed out (120s)")
        except Exception as e:
            print(f"[DriveSyncCallback] WARNING: sync failed: {e}")

    def on_trial_result(self, iteration, trials, trial, result, **info):
        if _time.time() - self._last_sync >= self._sync_interval:
            self._sync(reason=f"periodic, iter={result.get('training_iteration', '?')}")

    def on_trial_complete(self, iteration, trials, trial, **info):
        if _time.time() - self._last_sync >= 60:
            self._sync(reason="trial complete")

    def on_experiment_end(self, trials, **info):
        self._sync(reason="experiment end")


class BestTrialReporter(TuneCallback):
    """Periodically prints the best trial's config and metrics."""

    def __init__(self, metric="best_val_mca", mode="max", every_n_results=20):
        self._metric = metric
        self._mode = mode
        self._every_n = every_n_results
        self._result_count = 0
        self._best_value = float('-inf') if mode == "max" else float('inf')
        self._best_config = None
        self.best_epoch = 0

    def on_trial_result(self, iteration, trials, trial, result, **info):
        self._result_count += 1
        val = result.get(self._metric, None)
        if val is None:
            return
        improved = (val > self._best_value) if self._mode == "max" else (val < self._best_value)
        if improved:
            self._best_value = val
            self._best_config = trial.config.copy()
            self.best_epoch = iteration

        if self._result_count % self._every_n == 0 and self._best_config is not None:
            self._print_best(result)

    def _print_best(self, latest_result):
        print(f"\n{'---'*20}")
        print(f"  Best {self._metric}: {self._best_value*100:.2f}% "
              f"(after {self._result_count} results)")
        for k, v in self._best_config.items():
            if isinstance(v, float):
                print(f"    {k}: {v:.2e}" if abs(v) < 0.01 else f"    {k}: {v:.4f}")
            else:
                print(f"    {k}: {v}")
        print(f"  Best epoch: {self.best_epoch}")
        print(f"{'---'*20}")


ray.shutdown()
ray.init(
    ignore_reinit_error=True,
    runtime_env={
        "env_vars": {
            "CUDA_MPS_PIPE_DIRECTORY": "/tmp/nvidia-mps",
            "CUDA_MPS_LOG_DIRECTORY": "/tmp/nvidia-log",
            "CUDA_DEVICE_ORDER": "PCI_BUS_ID",
            "CUDA_VISIBLE_DEVICES": "0",
        }
    }
)

norm_stats = _load_norm_stats(LOCAL_DATASET_PATH)

print(f"Dataset: {LOCAL_DATASET_PATH}")
print(f"base_max_lr: {BASE_MAX_LR:.2e}")


# Define trainable
trainable = tune.with_resources(
    tune.with_parameters(
        train_linet_onecycle,
        data_root=LOCAL_DATASET_PATH,
        norm_stats=norm_stats,
        base_max_lr=BASE_MAX_LR,
        seed=SEED,
    ),
    resources={"cpu": 3, "gpu": 1.0 / 12},
)

# Callback to force-sync experiment state to Drive
drive_sync_cb = DriveSyncCallback(LOCAL_STORAGE_PATH, DRIVE_STORAGE_PATH, EXPERIMENT_NAME)


if RESUME_EXISTING:
    # =========================================================
    # RESUME: Restore previous experiment from Google Drive
    # =========================================================
    print("\n" + "=" * 60)
    print("RESUMING EXPERIMENT FROM GOOGLE DRIVE")
    print("=" * 60)

    tuner = tune.Tuner.restore(
        path=local_experiment_path,
        trainable=trainable,
        resume_unfinished=True,
        resume_errored=True,
    )

else:
    # =========================================================
    # NEW: Create fresh experiment
    # =========================================================
    print("\n" + "=" * 60)
    print("STARTING NEW ONECYCLE HPO EXPERIMENT")
    print("=" * 60)

    # Search space — base_max_lr is FIXED from LR Finder
    search_space = {                                                                                                       
        # OneCycle parameters                                 
        "pct_start": tune.quniform(0.20, 0.40, 0.05),
        "div_factor": tune.quniform(15.0, 35.0, 5.0),                                                                      
        "final_div_factor": tune.loguniform(1e3, 1e5),
                                                                                                                            
        # Stem — already aggressive at 22x under cosine;      
        # OneCycle amplifies the stem's cycle, so might settle lower                                                       
        "stem_lr_multiplier": tune.quniform(12.0, 25.0, 1.0),                                                              
                                                                                                                            
        # Regularization — REDUCED ranges (OneCycle is implicit regularizer)                                               
        "wd": tune.loguniform(5e-5, 5e-4),          # was [3e-5, 6e-4], tighten around 2.96e-4                             
        "dropout_p": tune.quniform(0.15, 0.35, 0.01), # was [0.25, 0.50], lower floor since best was 0.26                  
        "label_smoothing": tune.quniform(0.10, 0.20, 0.01), # was [0.05, 0.15], shift up since best was 0.18               
        "grad_clip_norm": tune.quniform(0.5, 1.5, 0.05),  # tighter — OneCycle peak LR causes gradient spikes              
                                                                                                                            
        # Augmentation (keep same — orthogonal to scheduler)                                                               
        "rgb_aug_prob": tune.quniform(0.8, 1.2, 0.01),                                                                     
        "rgb_aug_mag": tune.quniform(0.8, 1.2, 0.01),                                                                      
        "depth_aug_prob": tune.quniform(0.8, 1.2, 0.01),                                                                   
        "depth_aug_mag": tune.quniform(0.8, 1.2, 0.01),
    } 

    reporter = CLIReporter(
        parameter_columns=[
            "pct_start",
            "div_factor",
            "final_div_factor",
            "stem_lr_multiplier",
            "wd",
            "dropout_p",
            "label_smoothing",
            "grad_clip_norm",
            "rgb_aug_prob",
            "rgb_aug_mag",
            "depth_aug_prob",
            "depth_aug_mag",
        ],
        metric_columns={
            "training_iteration": "iter",
            "best_val_mca": "best_val_mca",
            "best_train_mca": "best_train_mca",
            "best_accuracy": "best_accuracy",
            "composite": "composite",
            "best_epoch": "best_epoch",
        },
        max_report_frequency=30,
        print_intermediate_tables=True,
    )

    # OptunaSearch ONLY — no ASHA (incompatible with OneCycleLR)
    optuna_search = OptunaSearch(
        metric="best_val_mca",
        mode="max",
        sampler=TPESampler(n_startup_trials=15),
    )

    tuner = tune.Tuner(
        trainable,
        param_space=search_space,
        tune_config=tune.TuneConfig(
            search_alg=optuna_search,
            num_samples=NUM_SAMPLES,
            max_concurrent_trials=12,
        ),
        run_config=ray.tune.RunConfig(
            storage_path=LOCAL_STORAGE_PATH,
            name=EXPERIMENT_NAME,
            progress_reporter=reporter,
            verbose=1,
            callbacks=[drive_sync_cb, BestTrialReporter(every_n_results=20)],
        ),
    )


# Run (or resume) tuning
print("\n" + "=" * 60)
print("STARTING ONECYCLE HYPERPARAMETER TUNING")
print("=" * 60)

results = tuner.fit()

best_result = results.get_best_result("best_val_mca", "max")

print("\n" + "=" * 60)
print("TUNING COMPLETE")
print("=" * 60)
print(f"Best Trial Config: {best_result.config}")
print(f"Best Trial Val MCA: {best_result.metrics['best_val_mca']:.4f}")
print(f"Best Trial Accuracy: {best_result.metrics['best_accuracy']:.4f}")
print(f"Best Trial Loss: {best_result.metrics['best_loss']:.4f}")
print(f"\nExperiment saved to: {local_experiment_path}")
print(f"Drive backup: {experiment_path}")
print(f"To resume after Colab dies: just re-run this notebook.")

In [ ]:
# =============================================================================
# SAVE RESULTS CSV (for offline analysis)
# =============================================================================

import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

results_df = results.get_dataframe()

csv_dir = f"{DRIVE_STORAGE_PATH}/analysis"
Path(csv_dir).mkdir(parents=True, exist_ok=True)

csv_path = f"{csv_dir}/sun_onecycle_hpo_results_{timestamp}.csv"
results_df.to_csv(csv_path, index=False)
print(f"Results CSV saved: {csv_path}")
print(f"  Trials: {len(results_df)}")

latest_path = f"{csv_dir}/sun_onecycle_hpo_results_latest.csv"
results_df.to_csv(latest_path, index=False)
print(f"Latest copy: {latest_path}")

In [ ]:
# Analyze Top 10 Trials from OneCycle HPO (ranked by best_val_mca)
import pandas as pd

print("=" * 80)
print("TOP 10 TRIALS BY BEST VAL MCA (ONECYCLE HPO)")
print("=" * 80)

df = results.get_dataframe()
df_sorted = df.sort_values('best_val_mca', ascending=False)

display_cols = [
    'best_val_mca', 'best_train_mca', 'best_accuracy', 'best_train_acc', 'composite', 'best_epoch',
    'config/pct_start',
    'config/div_factor',
    'config/final_div_factor',
    'config/stem_lr_multiplier',
    'config/wd',
    'config/dropout_p',
    'config/label_smoothing',
    'config/grad_clip_norm',
    'config/rgb_aug_prob',
    'config/rgb_aug_mag',
    'config/depth_aug_prob',
    'config/depth_aug_mag',
]

top_10 = df_sorted[display_cols].head(10)

top_10_formatted = top_10.copy()
top_10_formatted['best_val_mca'] = top_10_formatted['best_val_mca'].apply(lambda x: f"{x*100:.2f}%")
top_10_formatted['best_train_mca'] = top_10_formatted['best_train_mca'].apply(lambda x: f"{x*100:.2f}%")
top_10_formatted['best_accuracy'] = top_10_formatted['best_accuracy'].apply(lambda x: f"{x*100:.2f}%")

sci_cols = ['config/wd', 'config/final_div_factor']
for col in sci_cols:
    if col in top_10_formatted.columns:
        top_10_formatted[col] = top_10_formatted[col].apply(lambda x: f"{x:.2e}")

float_cols = [c for c in display_cols if c.startswith('config/') and c not in sci_cols]
for col in float_cols:
    if col in top_10_formatted.columns:
        top_10_formatted[col] = top_10_formatted[col].apply(lambda x: f"{x:.3f}")

print(top_10_formatted.to_string(index=False))
print("\n" + "=" * 80)

In [ ]:
# =============================================================================
# ANALYZE TOP 10 TRIALS BY BEST VAL MCA
# =============================================================================

import pandas as pd

df = results.get_dataframe()
df = df.sort_values("best_val_mca", ascending=False)
top_10 = df.head(10).copy()

config_cols = [c for c in df.columns if c.startswith("config/")]

print("=" * 80)
print("TOP 10 TRIALS BY BEST VAL MCA")
print("=" * 80)

for rank, (_, row) in enumerate(top_10.iterrows(), 1):
    gap = row.get("best_train_mca", 0) - row.get("best_val_mca", 0)
    print(f"\n--- #{rank} | Best Val MCA: {row['best_val_mca']*100:.2f}% | "
          f"Acc: {row['best_accuracy']*100:.2f}% | "
          f"Gap: {gap*100:.1f}pp ---")

print("\n" + "=" * 80)
print("HYPERPARAMETER RANGES ACROSS TOP 10")
print("=" * 80)
print(f"{'Parameter':<35} {'Min':>12} {'Max':>12} {'Median':>12}")
print("-" * 75)

for col in config_cols:
    short_name = col.replace("config/", "")
    col_min = top_10[col].min()
    col_max = top_10[col].max()
    col_med = top_10[col].median()
    if abs(col_med) < 0.001:
        print(f"{short_name:<35} {col_min:>12.2e} {col_max:>12.2e} {col_med:>12.2e}")
    else:
        print(f"{short_name:<35} {col_min:>12.4f} {col_max:>12.4f} {col_med:>12.4f}")

In [ ]:
# =============================================================================
# FULL CONFIG TABLE -- TOP 10 BY BEST VAL MCA
# =============================================================================

import pandas as pd

df = results.get_dataframe()
df = df.sort_values("best_val_mca", ascending=False)
top_10 = df.head(10).copy()

config_cols = [c for c in df.columns if c.startswith("config/")]
top_10["gap"] = top_10.get("best_train_mca", 0) - top_10.get("best_val_mca", 0)

display_df = top_10[["best_val_mca", "best_accuracy", "training_iteration"] + config_cols].copy()
display_df.insert(0, "rank", range(1, len(display_df) + 1))
display_df["best_val_mca"] = display_df["best_val_mca"].apply(lambda x: f"{x*100:.2f}%")
display_df["best_accuracy"] = display_df["best_accuracy"].apply(lambda x: f"{x*100:.2f}%")
display_df["training_iteration"] = display_df["training_iteration"].astype(int)

sci_cols = [c for c in config_cols if any(k in c for k in ["wd", "final_div"]) and "multiplier" not in c]
float_cols = [c for c in config_cols if c not in sci_cols]

for col in sci_cols:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda x: f"{x:.2e}")
for col in float_cols:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}")

display_df.columns = [c.replace("config/", "") for c in display_df.columns]

print(display_df.to_string(index=False))